# Circuit rewiring example

Adds a new **edge population** to a SONATA circuit: *which* neurons are connected is drawn from a
structural connectivity model per pathway, and every connection (synapse) gets


In [ ]:
from datetime import datetime
from pathlib import Path

from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask

from obi_one.scientific.blocks.distributions.uniform import IntUniformDistribution
from obi_one.scientific.blocks.morphology_locations.random import RandomMorphologyLocations
from obi_one.scientific.blocks.neuron_sets.population import VirtualPopulationNeuronSet
from obi_one.scientific.blocks.neuron_sets.predefined import BiophysicalPopulationPredefinedNeuronSet
from obi_one.scientific.blocks.neuron_sets.property import BiophysicalPopulationPropertyNeuronSet
from obi_one.scientific.blocks.neuron_sets.property import NeuronPropertyFilter
from obi_one.scientific.blocks.neuron_sets.property import PointPopulationPropertyNeuronSet
from obi_one.scientific.blocks.neuron_sets.specific import AllBiophysicalNeurons
from obi_one.scientific.blocks.structural_conn_models.distance_dependent import DistanceDependentConnModel
from obi_one.scientific.blocks.structural_conn_models.erdos_renyi import ErdosRenyiConnModel
from obi_one.scientific.blocks.synaptic_models.tsodyks_markram.block import (
    ExcitatoryTsodyksMarkramSynapticModel,
    InhibitoryTsodyksMarkramSynapticModel,
)
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.circuit_rewiring import (
    CircuitConnectivityPathway,
    CircuitRewiringScanConfig,
)
import conntility
import numpy
from obi_one.scientific.blocks.distributions.discrete import IntDiscreteDistribution

In [ ]:
input_root = Path("/Users/mwr/Documents/artefacts/SONATA/PD14-test/PD14_PD14Network---second-test")
# A fresh folder per run, so re-running the notebook never collides with earlier output
output_root = Path("../../../../../../obi-output/circuit_rewiring") / datetime.now().strftime("%Y%m%d_%H%M%S")

Read reference connectome and calculate statistics

In [ ]:
mat = conntility.ConnectivityMatrix.from_h5(input_root / "../connectivity_matrix.h5")

bins = numpy.arange(1, 21)
H = numpy.histogram(mat.matrix.data, bins=bins)[0]
H = H / H.sum()
msk = H > 0

syns_con = IntDiscreteDistribution(values=tuple(bins[:-1][msk]), probabilities=tuple(H[msk]))
mat = mat.compress()

pw_mat = mat.condense("mtype")
lst_mtypes = pw_mat.vertices["mtype"]

nrn_counts = mat.vertices["mtype"].value_counts()[lst_mtypes].to_numpy()
n_pairs = nrn_counts.reshape((-1, 1)) * nrn_counts.reshape((1, -1))

p_mat = pw_mat.array / n_pairs

Build campaign

In [ ]:
config = CircuitRewiringScanConfig.empty_config()

config.set(
    Info(campaign_name="Rewire example: Make distance-dependent", campaign_description="We make the PD14 model distance-dependent"),
    name="info",
)
config.add(syns_con, name="Synapses per connection")

exc_model = ExcitatoryTsodyksMarkramSynapticModel()
inh_model = InhibitoryTsodyksMarkramSynapticModel()
config.add(exc_model, name="ExcModel")
config.add(inh_model, name="InhModel")

for mtype in lst_mtypes:
    n_set = PointPopulationPropertyNeuronSet(population="microcircuit",
                                            property_filter=NeuronPropertyFilter(
                                                filter_dict={"mtype": [mtype]}
                                            ))
    config.add(n_set, name=mtype)

for i, src in enumerate(lst_mtypes):
    src_set = config.neuron_sets[src]
    is_inh = "I" in src
    if is_inh:
        used_model = inh_model
    else:
        used_model = exc_model
    for j, tgt in enumerate(lst_mtypes):
        tgt_set = config.neuron_sets[tgt]
        if p_mat[i, j] == 0:
            continue
        con_mdl = DistanceDependentConnModel(scale_factor=numpy.minimum(p_mat[i, j] * 3, 1.0),
                                             length_scale=50.0)
        config.add(con_mdl, name=(src + "-" + tgt + "_model"))

        config.pathways[(src + "-" + tgt + "_pathway")] = CircuitConnectivityPathway(
            source_neuron_set=src_set.ref,
            conn_model=con_mdl.ref,
            synapses_per_connection=syns_con.ref,
            synaptic_model=used_model.ref,
            target=tgt_set.ref,
        )





In [ ]:
from obi_auth import get_token
from obi_notebook.get_projects import get_projects
from entitysdk import Client, models

env = "staging"

token = get_token(environment=env, auth_mode="daf")
project = get_projects(token=token, env=env)

In [ ]:
client = Client(project_context=project, token_manager=token, environment=env)

circ = client.search_entity(entity_type=models.Circuit, query={
    "name__ilike": "PD14Network---second"
}).one()

In [ ]:
from obi_one.scientific.from_id.circuit_from_id import CircuitFromID

circ = CircuitFromID(id_str=str(circ.id))

config.set(
    CircuitRewiringScanConfig.Initialize(
        circuit=circ,
        edge_population_name="rewired",  # prefix of the new edge population name(s)
        edge_population_type="chemical",
        seed=0,
    ),
    name="initialize",
)

# Validated config
validated_config = config.validated_config()

# Generate the (single-instance) scan
grid_scan = GridScanGenerationTask(
    form=validated_config, output_root=str(output_root), coordinate_directory_option="ZERO_INDEX"
)
grid_scan.execute()

EXECUTE!

In [ ]:
run_tasks_for_generated_scan(grid_scan, db_client=client)


### Inspect the result

The rewired circuit is written to `<output_root>/0/SONATA`. The new edge population(s) were appended to its
`circuit_config.json`; the original edge populations are untouched.

In [ ]:
single_config = grid_scan.single_configs[0]
rewired = Circuit(name="rewired", path=str(single_config.coordinate_output_root / "SONATA" / "circuit_config.json"))
original = validated_config.initialize.circuit

original_edge_populations = set(original.sonata_circuit.edges.population_names)
print(f"Original edge populations: {sorted(original_edge_populations)}\n")
for name in rewired.sonata_circuit.edges.population_names:
    edges = rewired.sonata_circuit.edges[name]
    tag = "" if name in original_edge_populations else "  <-- new"
    print(f"{name}: {edges.source.name} -> {edges.target.name}, {edges.size} synapses{tag}")

In [ ]:
new_edge_populations = [
    name for name in rewired.sonata_circuit.edges.population_names if name not in original_edge_populations
]
for name in new_edge_populations:
    edges = rewired.sonata_circuit.edges[name]
    properties = sorted(p for p in edges.property_names if not p.startswith("@"))
    print(f"{name}\n  edge properties: {properties}\n")

In [ ]:
# A few synapses of the first new edge population: connectivity, postsynaptic location and parameters
edges = rewired.sonata_circuit.edges[new_edge_populations[0]]
table = edges.get(edges.ids(), properties=["@source_node", "@target_node", *sorted(p for p in edges.property_names if not p.startswith("@"))])
table.head()

### Check the pathways

The synaptic model of each pathway sets `syn_type_id` (excitatory vs. inhibitory), and the number of synapses per
connection follows the configured distributions.

In [ ]:
for name in new_edge_populations:
    edges = rewired.sonata_circuit.edges[name]
    table = edges.get(edges.ids(), properties=["@source_node", "@target_node", "syn_type_id"])
    n_connections = len(table[["@source_node", "@target_node"]].drop_duplicates())
    synapses_per_connection = table.groupby(["@source_node", "@target_node"]).size()
    print(f"{name}: {len(table)} synapses in {n_connections} connections "
          f"({synapses_per_connection.min()}-{synapses_per_connection.max()} synapses per connection)")
    print("  synapses per syn_type_id:", table["syn_type_id"].value_counts().to_dict())